# Sistema de diàleg basat en regles: extensió

## Llibreries

In [1]:
# Llibreries de preprocessament
import nltk
from nltk.tokenize.treebank import TreebankWordTokenizer
from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer

nltk.download("omw-1.4")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("punkt")
nltk.download("punkt_tab")

import json
import sys


[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\berta\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\berta\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\berta\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\berta\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\berta\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Classe base

In [10]:
class DialogueSystem:
    """Sistema de diàleg basat en regles per recomanar habitatges."""

    EXIT_COMMANDS = {
        "sortir", "surt", "adeu", "cancel lar", "cancel",
        "quit", "exit", "bye", "leave", "q", "salir", "adios",
    }

    ANY_COMMANDS = {
        "qualsevol",
        "indiferent",
        "tant me fa",
        "sense preferència",
        "sense preferencia",
        "em dona igual",
        "m'és igual",
        "any",
    }

    LOCATION_OPTIONS = [
        "L'Hospitalet de Llobregat",
        "Esplugues de Llobregat",
        "Barcelona",
        "Santa Coloma de Gramenet",
    ]

    def __init__(self, data_path):
        self.data = self.load_data(data_path)
        self.user_preferences = {}
        self.available_options = self.initialize_available_options()

    # ------------------------------------------------------------------
    # Dades
    # ------------------------------------------------------------------
    def load_data(self, path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)

    def initialize_available_options(self):
        options = {}

        for house in self.data["houses"]:
            for key, value in house.items():
                options.setdefault(key, set()).add(value)

        return options

    # ------------------------------------------------------------------
    # Comprensió (NLU)
    # ------------------------------------------------------------------
    def normalize_text(self, answer):
        import re
        import unicodedata

        text = unicodedata.normalize("NFD", str(answer).casefold())
        text = "".join(
            char
            for char in text
            if unicodedata.category(char) != "Mn"
        )
        text = re.sub(r"[^\w\s]", " ", text)
        text = text.replace("_", " ")

        return " ".join(text.split())

    def preprocess_answer(self, answer):
        # Conservem les quantitats sense separar-ne els components.
        return [str(answer)]

    def get_numerical_value(self, tok_answer):
        import re

        text = " ".join(tok_answer).casefold()

        pattern = (
            r"(?<!\w)[+-]?\d+"
            r"(?:(?:[.,]\d+)|(?:[ \u00a0]\d{3}(?!\d)))*"
            r"(?:\s*(?:miles|mil|k)\b)?"
            r"(?!\w)"
        )

        matches = list(re.finditer(pattern, text))

        if len(matches) != 1:
            return None

        return self.normalize_number(matches[0].group())

    def normalize_number(self, value):
        import re
        from decimal import Decimal, InvalidOperation

        if value is None:
            return None

        # Els valors que ja són numèrics no necessiten interpretar
        # separadors de milers.
        if isinstance(value, (int, float, Decimal)):
            number = Decimal(str(value))

            if not number.is_finite():
                return None

            return (
                int(number)
                if number == number.to_integral_value()
                else float(number)
            )

        text = str(value).strip().casefold()
        text = re.sub(r"[€$]", "", text).strip()
        text = re.sub(r"\s*(euros?|eur)\s*$", "", text).strip()

        multiplier = 1
        suffix = re.search(r"\s*(miles|mil|k)\s*$", text)

        if suffix:
            multiplier = 1000
            text = text[:suffix.start()].strip()

        text = re.sub(r"\s+", "", text)

        if not re.fullmatch(r"[+-]?\d+(?:[.,]\d+)*", text):
            return None

        if "." in text and "," in text:
            decimal_separator = (
                "."
                if text.rfind(".") > text.rfind(",")
                else ","
            )
            thousands_separator = (
                "," if decimal_separator == "." else "."
            )

            integer_part, fraction = text.rsplit(
                decimal_separator, 1
            )

            grouped_integer = (
                r"[+-]?\d{1,3}(?:"
                + re.escape(thousands_separator)
                + r"\d{3})+"
            )

            if not re.fullmatch(grouped_integer, integer_part):
                return None

            text = (
                integer_part.replace(thousands_separator, "")
                + "."
                + fraction
            )

        elif "." in text or "," in text:
            separator = "." if "." in text else ","

            grouped_number = (
                r"[+-]?\d{1,3}(?:"
                + re.escape(separator)
                + r"\d{3})+"
            )

            if re.fullmatch(grouped_number, text):
                text = text.replace(separator, "")
            elif text.count(separator) == 1:
                text = text.replace(separator, ".")
            else:
                return None

        try:
            number = Decimal(text) * multiplier
        except InvalidOperation:
            return None

        return (
            int(number)
            if number == number.to_integral_value()
            else float(number)
        )

    def normalize_boolean(self, value):
        if value is None or isinstance(value, bool):
            return value

        text = self.normalize_text(value)

        if text in {"si", "s", "yes", "y", "true", "1"}:
            return True

        if text in {"no", "n", "false", "0"}:
            return False

        return None

    def is_exit_command(self, answer):
        text = self.normalize_text(answer)

        return text in {
            self.normalize_text(command)
            for command in self.EXIT_COMMANDS
        }

    def is_any_answer(self, answer):
        text = self.normalize_text(answer)

        return text in {
            self.normalize_text(command)
            for command in self.ANY_COMMANDS
        }

    # ------------------------------------------------------------------
    # Generació (NLG)
    # ------------------------------------------------------------------
    def print_welcome(self):
        print("Benvingut/da a l'assistent de cerca d'habitatges!")
        print(
            "Pots escriure «sortir» per acabar la conversa "
            "o «qualsevol» si no tens preferència."
        )

    def print_goodbye(self):
        print("Gràcies per utilitzar l'assistent. Fins aviat!")

    def print_question(self, question, possible_options=None):
        print(question["question"])

        if possible_options:
            translations = {
                "sale": "comprar",
                "rent": "llogar",
                "yes": "sí",
                "no": "no",
                "any": "qualsevol",
            }

            options = [
                translations.get(str(option), str(option))
                for option in possible_options
            ]

            print("Opcions:", ", ".join(options))

    def print_no_houses(self):
        print(
            "No s'ha trobat cap habitatge "
            "que compleixi les teves preferències."
        )

    def print_house(self, house):
        operation = house.get("type")
        operation_label = {
            "sale": "Compra",
            "rent": "Lloguer",
        }.get(operation, "No especificat")

        price = self.normalize_number(house.get("price"))

        if price is None:
            price_text = "No especificat"
        else:
            unit = "€/mes" if operation == "rent" else "€"
            price_text = f"{price:g} {unit}"

        print(f"Operació: {operation_label}")
        print("Habitacions:", house.get("bedrooms", "No especificat"))
        print("Banys:", house.get("bathrooms", "No especificat"))
        print("Preu:", price_text)
        print(
            "Superfície:",
            house.get("square_meters", "No especificat"),
            "m²",
        )
        print("Ubicació:", house.get("location", "No especificat"))
        print("Planta:", house.get("floor", "No especificat"))

        for key, label in [
            ("terrace", "Terrassa"),
            ("elevator", "Ascensor"),
            ("commercial_use", "Ús comercial permès"),
        ]:
            value = self.normalize_boolean(house.get(key))

            description = (
                "Sí" if value is True
                else "No" if value is False
                else "No especificat"
            )

            print(f"{label}: {description}")

        print()

    def print_suitable_houses(self, suitable_houses):
        if not suitable_houses:
            self.print_no_houses()
            return

        print(
            "\nAquests són els habitatges "
            "que compleixen les teves preferències:"
        )

        for house in suitable_houses:
            self.print_house(house)

    # ------------------------------------------------------------------
    # Entrada d'usuari
    # ------------------------------------------------------------------
    def read_input(self, prompt):
        try:
            return input(prompt)
        except (KeyboardInterrupt, EOFError):
            self.exit_dialogue()

    def exit_dialogue(self):
        self.print_goodbye()
        sys.exit()

    def handle_special_answer(self, answer):
        if self.is_exit_command(answer):
            self.exit_dialogue()

        if self.is_any_answer(answer):
            return True, None

        return False, None

    # ------------------------------------------------------------------
    # Gestió de preguntes
    # ------------------------------------------------------------------
    def process_multichoice_question(self, question, options):
        self.print_question(question, options)

        type_answers = {
            "sale": "sale",
            "comprar": "sale",
            "compra": "sale",
            "vull comprar": "sale",
            "rent": "rent",
            "llogar": "rent",
            "lloguer": "rent",
            "vull llogar": "rent",
        }

        while True:
            answer = self.read_input(question["prompt"])

            special, value = self.handle_special_answer(answer)
            if special:
                return value

            text = self.normalize_text(answer)
            value = type_answers.get(text)

            if value not in options:
                value = next(
                    (
                        option
                        for option in options
                        if self.normalize_text(option) == text
                    ),
                    None,
                )

            if value is not None:
                return value

            print("Indica una de les opcions proposades.")

    def ask_income_question(self):
        return self.ask_numerical_question({
            "question": "Quins són els ingressos mensuals de la llar?",
            "prompt": "Ingressos mensuals (€/mes): ",
            "answer_key": "income",
        })

    def ask_floor_question(self):
        return self.ask_numerical_question({
            "question": "A quina planta mínima vols viure?",
            "prompt": "Planta mínima: ",
            "answer_key": "floor",
        })

    def ask_yes_no_any_question(self, question):
        self.print_question(question, ["sí", "no", "qualsevol"])

        while True:
            answer = self.read_input(question["prompt"])

            special, value = self.handle_special_answer(answer)
            if special:
                return value

            value = self.normalize_boolean(answer)

            if value is not None:
                return value

            print("Resposta invàlida. Escriu «sí», «no» o «qualsevol».")

    def ask_numerical_question(self, question):
        self.print_question(question)

        key = question["answer_key"]
        integer_required = key in {"bedrooms", "bathrooms", "floor"}
        minimum = 0 if key == "floor" else 1

        while True:
            answer = self.read_input(question["prompt"])

            special, value = self.handle_special_answer(answer)
            if special:
                return value

            value = self.normalize_number(
                self.get_numerical_value(
                    self.preprocess_answer(answer)
                )
            )

            valid = (
                value is not None
                and (
                    value >= minimum
                    if key == "floor"
                    else value > 0
                )
            )

            if valid:
                if not integer_required or float(value).is_integer():
                    return int(value) if integer_required else value

            if key == "floor":
                requirement = "un enter igual o superior a zero"
            elif integer_required:
                requirement = "un enter positiu"
            else:
                requirement = "una quantitat positiva"

            print(
                f"Indica {requirement}, sense unitats, "
                "o escriu «qualsevol»."
            )

    def ask_intent_question(self, question, default_intent="min"):
        import re
        self.print_question(question)
        key = question["answer_key"]
        
        while True:
            answer = self.read_input(question["prompt"])
            
            special, value = self.handle_special_answer(answer)
            if special:
                return (None, None) if value is None else value
                
            text = self.normalize_text(answer)
            
            if key == "floor" and re.search(r"(atic|ultim|a dalt de tot)", text):
                return (5, None)
            
            pattern = r"(?<!\w)[+-]?\d+(?:(?:[.,]\d+)|(?:[  ]\d{3}(?!\d)))*(?:\s*(?:miles|mil|k))?(?!\w)"
            matches = list(re.finditer(pattern, text))
            
            if len(matches) == 2 and re.search(r"(entre|de)", text) and re.search(r"(i|a|y|-|fins)", text):
                num1 = self.normalize_number(matches[0].group())
                num2 = self.normalize_number(matches[1].group())
                if num1 is not None and num2 is not None:
                    return (min(num1, num2), max(num1, num2))
                    
            num = self.normalize_number(
                self.get_numerical_value(self.preprocess_answer(answer))
            )
            
            if num is not None:
                is_exact = re.search(r"(exact|nomes|just|unicament)", text)
                is_max = re.search(r"(maxim|fins|no mes|menys|topall)", text)
                is_min = re.search(r"(minim|almenys|mes de|a partir)", text)
                is_aprox = re.search(r"(aprox|sobre|mes o menys|voltant|prop de|uns)", text)
                
                if is_aprox:
                    marge = num * 0.15
                    return (num - marge, num + marge)
                elif is_exact:
                    return (num, num)
                elif is_max:
                    return (None, num)
                elif is_min:
                    return (num, None)
                else:
                    return (num, None) if default_intent == "min" else (None, num)
            
            print("No ho he entès. Indica un número ('exactament 2'), un rang ('entre 2 i 4') o aproximats ('aprox 100').")


    def ask_questions(self):
        # Cal escollir una operació per interpretar el pressupost.
        while True:
            operation = self.process_multichoice_question(
                {
                    "question": "Vols comprar o llogar un habitatge?",
                    "prompt": "Resposta (comprar/llogar): ",
                },
                ["sale", "rent"],
            )

            if operation in {"sale", "rent"}:
                self.user_preferences["type"] = operation
                break

            print(
                "Per continuar, cal escollir «comprar» o «llogar»: "
                "el pressupost té un significat diferent en cada cas."
            )

        # Els ingressos només són rellevants per al lloguer.
        self.user_preferences["income"] = None

        if operation == "rent":
            print(
                "Per a aquesta cerca, el límit de lloguer "
                "serà el 35% dels ingressos mensuals indicats."
            )
            self.user_preferences["income"] = self.ask_income_question()

        # Ubicació.
        self.print_question({
            "question": "En quina ciutat prefereixes viure?",
        })

        # Es mostra explícitament per conservar les opcions
        # encara que una classe filla redefineixi print_question().
        print("Opcions:", ", ".join(self.LOCATION_OPTIONS))

        while True:
            import re
            answer = self.read_input("Ciutat: ")

            special, value = self.handle_special_answer(answer)
            if special:
                self.user_preferences["location"] = value
                break

            text = self.normalize_text(answer)
            location = None
            
            if re.search(r"(hospitalet|hospi)", text):
                location = "L'Hospitalet de Llobregat"
            elif re.search(r"(esplugues|esplu)", text):
                location = "Esplugues de Llobregat"
            elif re.search(r"(coloma|santa coloma|santaco)", text):
                location = "Santa Coloma de Gramenet"
            elif re.search(r"(barcelona|bcn|barna)", text):
                location = "Barcelona"

            if location is not None:
                self.user_preferences["location"] = location
                break

            print("Escriu una de les ciutats indicades (pots posar només 'esplugues', 'bcn', etc) o «qualsevol».")

        # Pressupost coherent amb l'operació escollida.
        if operation == "rent":
            budget_question = {
                "question": (
                    "Quin és el teu pressupost màxim mensual "
                    "per al lloguer?"
                ),
                "prompt": "Lloguer màxim (€/mes): ",
                "answer_key": "price",
            }
        else:
            budget_question = {
                "question": (
                    "Quin és el teu pressupost màxim "
                    "per comprar l'habitatge?"
                ),
                "prompt": "Pressupost màxim de compra (€): ",
                "answer_key": "price",
            }

        # El pressupost, per defecte és un MÀXIM
        self.user_preferences["price"] = self.ask_intent_question(
            budget_question, 
            default_intent="max"
        )

        # Si s'han indicat tots dos límits, s'aplica el més baix.
        if operation == "rent":
            income = self.normalize_number(
                self.user_preferences["income"]
            )

            # Extraiem el màxim de la tupla (posició 1)
            budget_pref = self.user_preferences["price"]
            budget = budget_pref[1] if isinstance(budget_pref, tuple) else None

            if income is not None:
                income_limit = income * 0.35

                effective_limit = (
                    min(budget, income_limit)
                    if budget is not None
                    else income_limit
                )

                print(
                    "Amb les preferències indicades, "
                    f"el lloguer màxim serà de {effective_limit:g} €/mes."
                )

        # Necessitats de l'habitatge (per defecte MÍNIM).
        questions = [
            {
                "question": "Quantes habitacions busques?",
                "prompt": "Habitacions: ",
                "answer_key": "bedrooms",
            },
            {
                "question": "Quants banys necessites?",
                "prompt": "Banys: ",
                "answer_key": "bathrooms",
            },
            {
                "question": (
                    "Quina superfície busques "
                    "en metres quadrats?"
                ),
                "prompt": "Superfície (m²): ",
                "answer_key": "square_meters",
            },
        ]

        for question in questions:
            key = question["answer_key"]
            self.user_preferences[key] = self.ask_intent_question(
                question, default_intent="min"
            )

        self.user_preferences["floor"] = self.ask_intent_question(
            {
                "question": "A quina planta vols viure?",
                "prompt": "Planta (ex: 0 per baixa): ",
                "answer_key": "floor",
            }, 
            default_intent="min"
        )

        # «No» significa que la característica no és obligatòria.
        for key, question, prompt in [
            (
                "terrace",
                "És imprescindible que l'habitatge tingui terrassa?",
                "Terrassa: ",
            ),
            (
                "elevator",
                "És imprescindible que l'edifici tingui ascensor?",
                "Ascensor: ",
            ),
            (
                "commercial_use",
                "Necessites que l'habitatge permeti un ús comercial?",
                "Ús comercial: ",
            ),
        ]:
            self.user_preferences[key] = (
                self.ask_yes_no_any_question({
                    "question": question,
                    "prompt": prompt,
                })
            )

        return self.user_preferences

    # ------------------------------------------------------------------
    # Cerca d'habitatges
    # ------------------------------------------------------------------
    def house_matches(self, house, key, preference):
        if preference is None:
            return True

        # Ignorem els rangs buits (None, None)
        if isinstance(preference, tuple) and all(v is None for v in preference):
            return True

        if key == "type":
            return house.get("type") == preference

        if key == "location":
            return (
                self.normalize_text(house.get("location", ""))
                == self.normalize_text(preference)
            )

        if key in {"price", "bedrooms", "bathrooms", "square_meters", "floor"}:
            actual = self.normalize_number(house.get(key))

            # Adaptació a tuples (min, max) o valor simple
            if isinstance(preference, tuple):
                min_val, max_val = preference
            else:
                min_val = None if key == "price" else preference
                max_val = preference if key == "price" else None

            min_val = self.normalize_number(min_val)
            max_val = self.normalize_number(max_val)

            if actual is None:
                return False
            if min_val is not None and actual < min_val:
                return False
            if max_val is not None and actual > max_val:
                return False
            return True

        if key == "income":
            if house.get("type") != "rent":
                return True

            income = self.normalize_number(preference)
            price = self.normalize_number(house.get("price"))

            return (
                income is not None
                and price is not None
                and price <= income * 0.35
            )

        if key in {"terrace", "elevator", "commercial_use"}:
            return (
                preference is not True
                or self.normalize_boolean(house.get(key)) is True
            )

        return True

    
    def find_suitable_houses(self):
        suitable_houses = []

        for house in self.data["houses"]:
            if all(
                self.house_matches(house, key, preference)
                for key, preference in self.user_preferences.items()
            ):
                suitable_houses.append(house)

        return suitable_houses

    # ------------------------------------------------------------------
    # Flux principal
    # ------------------------------------------------------------------
    def run(self):
        self.user_preferences = {}
        self.print_welcome()
        self.ask_questions()

        suitable_houses = self.find_suitable_houses()
        self.print_suitable_houses(suitable_houses)

        return suitable_houses

## Execució

In [11]:
# Execució del sistema base
system = DialogueSystem("./house_data.json")
system.run()

Benvingut/da a l'assistent de cerca d'habitatges!
Pots escriure «sortir» per acabar la conversa o «qualsevol» si no tens preferència.
Vols comprar o llogar un habitatge?
Opcions: comprar, llogar
Indica una de les opcions proposades.
Indica una de les opcions proposades.
Indica una de les opcions proposades.
Per a aquesta cerca, el límit de lloguer serà el 35% dels ingressos mensuals indicats.
Quins són els ingressos mensuals de la llar?
En quina ciutat prefereixes viure?
Opcions: L'Hospitalet de Llobregat, Esplugues de Llobregat, Barcelona, Santa Coloma de Gramenet
Quin és el teu pressupost màxim mensual per al lloguer?
Amb les preferències indicades, el lloguer màxim serà de 700 €/mes.
Quantes habitacions busques?
Quants banys necessites?
Quina superfície busques en metres quadrats?
A quina planta vols viure?
És imprescindible que l'habitatge tingui terrassa?
Opcions: sí, no, qualsevol
Resposta invàlida. Escriu «sí», «no» o «qualsevol».
Resposta invàlida. Escriu «sí», «no» o «qualsevo

[]

## Tasques

### 1. Detecció de casos límit i millora

Prova de casos numèrics

In [12]:
system = DialogueSystem("./house_data.json")

casos_numerics = [
    ("3", 3),
    ("30000", 30000),
    ("tres", 3),
    ("35k", 35000),
    ("un pis de 35k euros", 35000),
    ("35.000€", 35000),
    ("35.000", 35000),
    ("35.000,00", 35000),
    ("35,000.00", 35000),
    ("35 mil", 35000),
    ("no ho sé", None),
    ("", None),
]

for entrada, esperat in casos_numerics:
    tokens = system.preprocess_answer(entrada)
    extret = system.get_numerical_value(tokens)

    try:
        resultat = system.normalize_number(extret)
        error = None
    except Exception as e:
        resultat = None
        error = f"{type(e).__name__}: {e}"

    correcte = error is None and resultat == esperat

    print(f"Entrada: {entrada!r}")
    print(f"  Tokens: {tokens}")
    print(f"  Valor extret: {extret!r}")
    print(f"  Valor normalitzat: {resultat}")
    print(f"  Valor esperat: {esperat}")
    print(f"  Error: {error}")
    print(f"  Resultat de la prova: {'OK' if correcte else 'FALLA'}")
    print()

Entrada: '3'
  Tokens: ['3']
  Valor extret: 3
  Valor normalitzat: 3
  Valor esperat: 3
  Error: None
  Resultat de la prova: OK

Entrada: '30000'
  Tokens: ['30000']
  Valor extret: 30000
  Valor normalitzat: 30000
  Valor esperat: 30000
  Error: None
  Resultat de la prova: OK

Entrada: 'tres'
  Tokens: ['tres']
  Valor extret: None
  Valor normalitzat: None
  Valor esperat: 3
  Error: None
  Resultat de la prova: FALLA

Entrada: '35k'
  Tokens: ['35k']
  Valor extret: 35000
  Valor normalitzat: 35000
  Valor esperat: 35000
  Error: None
  Resultat de la prova: OK

Entrada: 'un pis de 35k euros'
  Tokens: ['un pis de 35k euros']
  Valor extret: 35000
  Valor normalitzat: 35000
  Valor esperat: 35000
  Error: None
  Resultat de la prova: OK

Entrada: '35.000€'
  Tokens: ['35.000€']
  Valor extret: 35000
  Valor normalitzat: 35000
  Valor esperat: 35000
  Error: None
  Resultat de la prova: OK

Entrada: '35.000'
  Tokens: ['35.000']
  Valor extret: 35000
  Valor normalitzat: 35000
  V

En aquesta prova veiem que el nostre sistema de diàleg no és capaç de detectar cap format numèric excepte la indicació explícita del nombre en si

Prova de casos textuals

Provarem la sensibilitat del sistema de diàleg a majúscules, minúscules, accents i altres variants de les respostes textuals. Les proves que es realitzaran seran:

Per la pregunta de lloguer o compra:

| Prova         | Entrades        |
| ------------------ | --------------- |
| Opcions originals  | sale, rent      |
| Majúscules         | Sale, RENT      |
| Espais             | sale            |
| Puntuació          | sale.           |
| Entrada buida      |    | 

Només s'accepten les respostes textuals originals.

Per les preguntes de tipus sí/no:

| Prova         | Entrades                     |
| ------------------- | ---------------------------- | 
| Respostes bàsiques  | yes, no                      |
| Majúscules          | YES, No                      |
| Espais              | yes                          |
| Abreviatures        | y, s, n                      |
| Puntuació           | yes.                         | 
| Entrada buida       |               | 

El sistema ha estat capaç d'identificar les respostes bàsiques, les majúscules, els espais i les abreviatures. No ha estat capaç d'identificar la puntuació ni l'entrada buida.

Per les entrades de sortir del sistema i de qualsevol:

| Prova         | Entrades                     |
| ------------------- | ---------------------------- |
| Respostes bàsiques  | exit, any                    |
| Majúscules          | EXIT, Any                    |
| Espais              | exit                         |
| Puntuació           | exit.                        |

El sistema ha estat capaç d'identificar les respostes bàsiques, les majúscules, els espais però no la puntuació.

In [13]:
system.run()

Benvingut/da a l'assistent de cerca d'habitatges!
Pots escriure «sortir» per acabar la conversa o «qualsevol» si no tens preferència.
Vols comprar o llogar un habitatge?
Opcions: comprar, llogar
En quina ciutat prefereixes viure?
Opcions: L'Hospitalet de Llobregat, Esplugues de Llobregat, Barcelona, Santa Coloma de Gramenet
Quin és el teu pressupost màxim per comprar l'habitatge?
Quantes habitacions busques?
Quants banys necessites?
Quina superfície busques en metres quadrats?
A quina planta vols viure?
És imprescindible que l'habitatge tingui terrassa?
Opcions: sí, no, qualsevol
És imprescindible que l'edifici tingui ascensor?
Opcions: sí, no, qualsevol
Necessites que l'habitatge permeti un ús comercial?
Opcions: sí, no, qualsevol

Aquests són els habitatges que compleixen les teves preferències:
Operació: Compra
Habitacions: 3
Banys: 2
Preu: 360000 €
Superfície: 90 m²
Ubicació: Barcelona
Planta: 2
Terrassa: Sí
Ascensor: No
Ús comercial permès: No

Operació: Compra
Habitacions: 4
Bany

[{'id': 20,
  'type': 'sale',
  'bedrooms': '3',
  'bathrooms': '2',
  'price': '360k',
  'square_meters': '90',
  'floor': '2',
  'elevator': 'No',
  'commercial_use': 'No',
  'terrace': 'Yes',
  'location': 'Barcelona'},
 {'id': 21,
  'type': 'sale',
  'bedrooms': '4',
  'bathrooms': '2',
  'price': '480k',
  'square_meters': '120',
  'floor': '3',
  'elevator': 'Yes',
  'commercial_use': 'Yes',
  'terrace': 'Yes',
  'location': 'Barcelona'},
 {'id': 23,
  'type': 'sale',
  'bedrooms': '3',
  'bathrooms': '2',
  'price': '285k',
  'square_meters': '95',
  'floor': '1',
  'elevator': 'Yes',
  'commercial_use': 'No',
  'terrace': 'No',
  'location': 'Barcelona'},
 {'id': 24,
  'type': 'sale',
  'bedrooms': '1',
  'bathrooms': '1',
  'price': '200k',
  'square_meters': '50',
  'floor': '2',
  'elevator': 'No',
  'commercial_use': 'No',
  'terrace': 'Yes',
  'location': 'Barcelona'},
 {'id': 25,
  'type': 'sale',
  'bedrooms': '2',
  'bathrooms': '1',
  'price': '180k',
  'square_meters'

Per tant, volem millorar el sistema de diàleg de manera que:

- Pugui identificar qualsevol format numèric.
- Pugui identificar qualsevol format textual de les respostes.
- Sigui sensisble a la puntuació i a l'entrada buida.

A part d'això, per tal de donar-li certa naturalitat i flexibilitat al sistema, implementarem:

- Afegir repetició a l'usuari el que el sistema ha entès de la seva entrada, per tal de afegir-li naturalitat.

In [14]:
import re
import unicodedata
from decimal import Decimal, InvalidOperation


class ImprovedDialogueSystem(DialogueSystem):
    """Amplia la comprensió i el feedback sense canviar els filtres."""

    NUMBER_WORDS = {
        "zero": 0, "baixa": 0, "baixos": 0,
        "primer": 1, "primera": 1,
        "segon": 2, "segona": 2,
        "tercer": 3, "tercera": 3,
        "quart": 4, "quarta": 4,
        "cinque": 5, "cinquena": 5,
        "sise": 6, "sisena": 6,
        "sete": 7, "setena": 7,
        "vuite": 8, "vuitena": 8,
        "nove": 9, "novena": 9,
        "dese": 10, "desena": 10,
        "un": 1, "una": 1, "u": 1, "uno": 1,
        "dos": 2, "dues": 2,
        "tres": 3,
        "quatre": 4, "cuatro": 4,
        "cinc": 5, "cinco": 5,
        "sis": 6, "seis": 6,
        "set": 7, "siete": 7,
        "vuit": 8, "ocho": 8,
        "nou": 9, "nueve": 9,
        "deu": 10, "diez": 10,
        "onze": 11, "once": 11,
        "dotze": 12, "doce": 12,
        "tretze": 13, "trece": 13,
        "catorze": 14, "catorce": 14,
        "quinze": 15, "quince": 15,
        "setze": 16,
        "disset": 17,
        "divuit": 18,
        "dinou": 19,
        "vint": 20, "veinte": 20,
        "trenta": 30, "treinta": 30,
        "quaranta": 40, "cuarenta": 40,
        "cinquanta": 50, "cincuenta": 50,
        "seixanta": 60, "sesenta": 60,
        "setanta": 70, "setenta": 70,
        "vuitanta": 80, "ochenta": 80,
        "noranta": 90, "noventa": 90,
    }

    ANY_ANSWERS = {
        "any", "qualsevol", "indiferent",
        "m es igual", "mes igual", "em dona igual",
        "m es indiferent", "tant me fa", "sense preferencia",
        "me da igual",
    }

    TYPE_ANSWERS = {
        "sale": "sale",
        "comprar": "sale",
        "compra": "sale",
        "vull comprar": "sale",
        "vull una casa de compra": "sale",
        "rent": "rent",
        "llogar": "rent",
        "lloguer": "rent",
        "alquilar": "rent",
        "vull llogar": "rent",
        "vull una casa de lloguer": "rent",
    }

    YES_ANSWERS = {
        "yes", "y", "si", "s", "true", "1",
        "si en vull", "en vull", "si ho vull",
        "si el necessito", "si la necessito",
        "vale", "ok", "d acord", "dacord", "perfecte", "i tant", "clar",
    }

    NO_ANSWERS = {
        "no", "n", "false", "0",
        "no en vull", "no cal", "que no cal",
        "no en necessito", "no ho necessito",
        "tampoc", "tampoc cal", "tampoc en vull",
    }

    # ------------------------------------------------------------------
    # Normalització textual
    # ------------------------------------------------------------------
    def normalize_text(self, answer):
        text = unicodedata.normalize("NFD", str(answer).casefold())
        text = "".join(
            char for char in text
            if unicodedata.category(char) != "Mn"
        )
        text = re.sub(r"[^\w\s]", " ", text)
        text = text.replace("_", " ")
        return " ".join(text.split())

    def is_exit_command(self, answer):
        # Només una resposta completa es considera una ordre.
        return self.normalize_text(answer) in self.EXIT_COMMANDS

    def is_any_answer(self, answer):
        return self.normalize_text(answer) in self.ANY_ANSWERS

    def normalize_boolean(self, value):
        import re
        if value is None or isinstance(value, bool):
            return value

        text = self.normalize_text(value)

        if text in self.YES_ANSWERS or re.search(r"\b(s+i+|y+e+s+|v+a+l+e+|o+k+|c+l+a+r+|i+\s*t+a+n+t+)\b", text):
            return True

        if text in self.NO_ANSWERS or re.search(r"\b(n+o+|t+a+m+p+o+c+)\b", text):
            return False

        return None

    def handle_special_answer(self, answer):
        if self.is_exit_command(answer):
            self.exit_dialogue()

        if self.is_any_answer(answer):
            print("Entesos, no fixem cap preferència per a aquest camp.")
            return True, None

        return False, None

    # ------------------------------------------------------------------
    # Extracció i normalització numèrica
    # ------------------------------------------------------------------
    def preprocess_answer(self, answer):
        # Conservem el text per no separar quantitats com "35.000€".
        return [str(answer)]

    def normalize_number(self, value):
        if value is None:
            return None

        if isinstance(value, (int, float, Decimal)):
            text = str(value)
        else:
            text = str(value).strip().casefold()

        text = re.sub(r"[€$]", "", text).strip()
        text = re.sub(r"\s*(euros?|eur)\s*$", "", text).strip()

        multiplier = 1
        suffix = re.search(r"\s*(k|mil|miles)\s*$", text)

        if suffix:
            multiplier = 1000
            text = text[:suffix.start()].strip()

        # Permet separadors de milers amb espais.
        text = re.sub(r"\s+", "", text)

        if not re.fullmatch(r"[+-]?\d+(?:[.,]\d+)*", text):
            return None

        if "." in text and "," in text:
            # L'últim separador representa els decimals.
            decimal_separator = (
                "." if text.rfind(".") > text.rfind(",") else ","
            )
            thousands_separator = "," if decimal_separator == "." else "."
            integer_part, fraction = text.rsplit(decimal_separator, 1)

            grouped_integer = (
                r"[+-]?\d{1,3}(?:"
                + re.escape(thousands_separator)
                + r"\d{3})+"
            )

            if not re.fullmatch(grouped_integer, integer_part):
                return None

            text = (
                integer_part.replace(thousands_separator, "")
                + "." + fraction
            )

        elif "." in text or "," in text:
            separator = "." if "." in text else ","
            grouped = (
                r"[+-]?\d{1,3}(?:"
                + re.escape(separator)
                + r"\d{3})+"
            )

            if re.fullmatch(grouped, text):
                # Convenció: "35.000" i "35,000" són 35000.
                text = text.replace(separator, "")
            elif text.count(separator) == 1:
                text = text.replace(separator, ".")
            else:
                return None

        try:
            number = Decimal(text) * multiplier
        except InvalidOperation:
            return None

        # No trunquem decimals: planta valida després si és enter.
        return int(number) if number == number.to_integral_value() else float(number)

    def get_numerical_value(self, tok_answer):
        text = " ".join(tok_answer).casefold()

        # Nombres amb dígits, separadors i multiplicador opcional.
        pattern = (
            r"(?<!\w)[+-]?\d+"
            r"(?:(?:[.,]\d+)|(?:[ \u00a0]\d{3}(?!\d)))*"
            r"(?:\s*(?:k|mil|miles)\b)?"
            r"(?!\w)"
        )

        matches = list(re.finditer(pattern, text))

        if matches:
            # No escollim arbitràriament si hi ha diverses quantitats.
            if len(matches) != 1:
                return None
            return self.normalize_number(matches[0].group())

        # Suport limitat per a nombres escrits amb paraules:
        # "tres", "trenta-cinc", "tres mil", "trenta-cinc mil".
        words = self.normalize_text(text).split()
        candidates = []

        for start in range(len(words)):
            word = words[start]

            if word == "mil":
                candidates.append(1000)
                continue

            if word not in self.NUMBER_WORDS:
                continue

            number = self.NUMBER_WORDS[word]
            end = start + 1

            if number >= 20 and number % 10 == 0:
                next_word = end
                if next_word < len(words) and words[next_word] in {"i", "y"}:
                    next_word += 1

                if (
                    next_word < len(words)
                    and words[next_word] in self.NUMBER_WORDS
                    and 1 <= self.NUMBER_WORDS[words[next_word]] <= 9
                ):
                    number += self.NUMBER_WORDS[words[next_word]]
                    end = next_word + 1

            if end < len(words) and words[end] in {"mil", "miles"}:
                number *= 1000
                end += 1

            candidates.append((start, end, number))

        # Evita comptar per separat les paraules d'un mateix nombre.
        groups = []
        covered_until = -1

        for candidate in candidates:
            if isinstance(candidate, int):
                continue

            start, end, number = candidate
            if start >= covered_until:
                groups.append(number)
                covered_until = end

        if groups:
            return groups[0] if len(groups) == 1 else None

        if words == ["mil"]:
            return 1000

        return None

    # ------------------------------------------------------------------
    # Preguntes amb ajuda i confirmació
    # ------------------------------------------------------------------
    def process_multichoice_question(self, question, options):
        self.print_question(question, options)

        while True:
            import re
            answer = self.read_input(question["prompt"])

            if not answer.strip():
                print("No has escrit cap resposta. Indica comprar, llogar o any.")
                continue

            special, value = self.handle_special_answer(answer)
            if special:
                return value

            text = self.normalize_text(answer)
            
            value = None
            if re.search(r"(comprar|compra|sale)", text):
                value = "sale"
            elif re.search(r"(llogar|lloguer|rent|alquilar)", text):
                value = "rent"

            if value is None:
                value = next(
                    (option for option in options
                     if self.normalize_text(option) == text),
                    None,
                )

            if value in options:
                description = {"sale": "comprar", "rent": "llogar"}.get(
                    value, str(value)
                )
                print(f"Entesos, vols {description}.")
                return value

            print("No ho he entès. Escriu comprar/sale, llogar/rent o any.")

    def ask_income_question(self):
        self.print_question({
            "question": "Quins són els ingressos mensuals de la llar?",
        })

        while True:
            answer = self.read_input("Ingressos mensuals: ")

            if not answer.strip():
                print("La resposta és buida. Escriu, per exemple, 3000 o 3k.")
                continue

            special, value = self.handle_special_answer(answer)
            if special:
                return value

            value = self.get_numerical_value(self.preprocess_answer(answer))

            if value is not None and value > 0:
                print(f"Entesos, els ingressos mensuals són {value:g} euros.")
                return value

            print(
                "Indica una única quantitat positiva, "
                "per exemple: 3000, 3.000, 3k o tres mil. "
                "També pots escriure any."
            )

    def ask_floor_question(self):
        self.print_question({
            "question": "A quina planta mínima vols viure? (o any)",
        })

        while True:
            answer = self.read_input("Planta: ")

            if not answer.strip():
                print("La resposta és buida. Escriu, per exemple, 3 o tres.")
                continue

            special, value = self.handle_special_answer(answer)
            if special:
                return value

            value = self.get_numerical_value(self.preprocess_answer(answer))

            if value is not None and value >= 0 and float(value).is_integer():
                value = int(value)
                print(f"Entesos, busquem a partir de la planta {value}.")
                return value

            print("Indica una planta entera no negativa, com 0, 3 o tres; o any.")

    def ask_yes_no_any_question(self, question):
        self.print_question(question, ["yes", "no", "any"])

        while True:
            answer = self.read_input(question["prompt"])

            if not answer.strip():
                print("La resposta és buida. Escriu sí, no o any.")
                continue

            special, value = self.handle_special_answer(answer)
            if special:
                return value

            value = self.normalize_boolean(answer)

            if value is not None:
                if value:
                    print(f"Entesos, és un requisit: {question['question']}")
                else:
                    print(
                        f"Entesos, no és un requisit: {question['question']}"
                    )
                return value

            print("No ho he entès. Escriu sí/yes, no o any.")

In [15]:
system = ImprovedDialogueSystem("./house_data.json")
system.run()

Benvingut/da a l'assistent de cerca d'habitatges!
Pots escriure «sortir» per acabar la conversa o «qualsevol» si no tens preferència.
Vols comprar o llogar un habitatge?
Opcions: comprar, llogar
Entesos, vols comprar.
En quina ciutat prefereixes viure?
Opcions: L'Hospitalet de Llobregat, Esplugues de Llobregat, Barcelona, Santa Coloma de Gramenet
Quin és el teu pressupost màxim per comprar l'habitatge?
Quantes habitacions busques?
Quants banys necessites?
Quina superfície busques en metres quadrats?
A quina planta vols viure?
És imprescindible que l'habitatge tingui terrassa?
Opcions: sí, no, qualsevol
Entesos, és un requisit: És imprescindible que l'habitatge tingui terrassa?
És imprescindible que l'edifici tingui ascensor?
Opcions: sí, no, qualsevol
Entesos, no és un requisit: És imprescindible que l'edifici tingui ascensor?
Necessites que l'habitatge permeti un ús comercial?
Opcions: sí, no, qualsevol
Entesos, és un requisit: Necessites que l'habitatge permeti un ús comercial?

Aque

[{'id': 13,
  'type': 'sale',
  'bedrooms': '4',
  'bathrooms': '2',
  'price': '330k',
  'square_meters': '150',
  'floor': '2',
  'elevator': 'Yes',
  'commercial_use': 'Yes',
  'terrace': 'Yes',
  'location': "L'Hospitalet de Llobregat"}]

## 2. Relaxació de restriccions i implementació de nous camins

Ara mateix tenim un sistema que és capaç de manejar molt bé la variabilitat de les respostes textuals, però no és capaç d'oferir opcions alternatives en el casos que el match de les respostes no coincideixi amb cap de les opcions que el sistema ofereix. Per tant, volem relaxar les restriccions del sistema:

- Si no coincideix amb cap de les restriccions, oferir les opcions alternatives que compleixin amb N-1 de les restriccions, i indicar quina s'ha relaxat. I si tampoc és possible, oferir les opcions que compleixin amb N-2 de les restriccions.

A part d'això, volem que el sistema sigui una mica més humà:

- Podem afegir un resum de les preferències abans de finalitzar la cerca i que l'usuari pugui confirmar si són correctes o no. Si no són correctes, oferir la possibilitat de tornar a començar la cerca.
- Missatge de comiat quan se surt del sistema amb quit.

Com que tenim un sistema suficientment flexible, podem eliminar la indicació de les opcions que el sistema ofereix a l'usuari, ja que el sistema és capaç de detectar qualsevol format textual de les respostes. Per tant, eliminarem la indicació de les opcions que el sistema ofereix a l'usuari.

In [18]:
class FlexibleDialogueSystem(ImprovedDialogueSystem):
    """Afegeix confirmació i cerca amb relaxació progressiva."""

    BOOLEAN_KEYS = {"terrace", "elevator", "commercial_use"}

    def __init__(self, data_path):
        super().__init__(data_path)
        self.search_details = []
        self.relaxation_level = None
        self.active_count = 0

    # ------------------------------------------------------------------
    # Descripció de les preferències
    # ------------------------------------------------------------------
    def describe_preference(self, key, value):

        if value is None or (isinstance(value, tuple) and all(v is None for v in value)):
            return "Sense preferència"
        
        if key == "type":
            return {"sale": "Comprar", "rent": "Llogar"}.get(value, str(value))
        
        if key == "income":
            income = self.normalize_number(value)
            limit = income * 0.35
            return f"Ingressos de {income:g} €/mes (lloguer màxim: {limit:g} €/mes)"
        
        if key in {"bedrooms", "bathrooms", "square_meters", "floor", "price"}:
            if isinstance(value, tuple):
                min_val, max_val = value
            else:
                min_val, max_val = (None, value) if key == "price" else (value, None)
            
            unit = " €" if key == "price" else " m²" if key == "square_meters" else ""
            prefix = {"price": "Pressupost", "bedrooms": "Habitacions", "bathrooms": "Banys", "square_meters": "Superfície", "floor": "Planta"}[key]
            
            if min_val is not None and max_val is not None:
                if min_val == max_val:
                    return f"{prefix}: exactament {min_val:g}{unit}"
                return f"{prefix}: entre {min_val:g} i {max_val:g}{unit}"
            
            elif min_val is not None:
                return f"{prefix}: mínim {min_val:g}{unit}"
            
            else:
                return f"{prefix}: màxim {max_val:g}{unit}"
            
        labels = {"terrace": "Terrassa", "elevator": "Ascensor", "commercial_use": "Ús comercial"}

        if key in labels:
            return f"{labels[key]}: obligatori" if value is True else f"{labels[key]}: no és un requisit"
        
        return f"{key}: {value}"

    
    def print_preferences_summary(self):
        print("\nAbans de cercar, aquestes són les teves preferències:")

        order = [
            "type", "income", "location", "price",
            "bedrooms", "bathrooms", "square_meters",
            "floor", "terrace", "elevator", "commercial_use",
        ]

        labels = {
            "type": "Tipus", "income": "Ingressos", "location": "Ubicació",
            "price": "Pressupost", "bedrooms": "Habitacions", "bathrooms": "Banys",
            "square_meters": "Superfície", "floor": "Planta", "terrace": "Terrassa",
            "elevator": "Ascensor", "commercial_use": "Ús comercial",
        }

        for key in order:
            value = self.user_preferences.get(key)
            if key == "income" and self.user_preferences.get("type") != "rent":
                continue
            if value is None or (isinstance(value, tuple) and all(v is None for v in value)):
                print(f"- {labels[key]}: sense preferència")
            else:
                print("-", self.describe_preference(key, value))

    # ------------------------------------------------------------------
    # Confirmació i reinici
    # ------------------------------------------------------------------
    def ask_confirmation(self, question):
        print(question)

        while True:
            answer = self.read_input("Resposta (sí/no): ")

            if self.is_exit_command(answer):
                self.exit_dialogue()

            if not answer.strip():
                print("La resposta és buida. Escriu sí o no.")
                continue

            value = self.normalize_boolean(answer)

            if value is not None:
                return value

            print("Cal una confirmació explícita. Escriu sí o no.")

    # ------------------------------------------------------------------
    # Restriccions actives
    # ------------------------------------------------------------------
    def get_active_preferences(self):
        active = {}

        for key, value in self.user_preferences.items():
            if value is None or (isinstance(value, tuple) and all(v is None for v in value)):
                continue

            if key in self.BOOLEAN_KEYS and value is not True:
                continue

            active[key] = value
            
        return active

    # ------------------------------------------------------------------
    # Cerca progressiva
    # ------------------------------------------------------------------
    def find_suitable_houses(self):
        active = self.get_active_preferences()

        self.active_count = len(active)
        self.search_details = []
        self.relaxation_level = None

        evaluated = []

        for house in self.data["houses"]:
            relaxed = [
                key
                for key, preference in active.items()
                if not self.house_matches(house, key, preference)
            ]

            evaluated.append({
                "house": house,
                "relaxed": relaxed,
                "matched": self.active_count - len(relaxed),
            })

        # 0: totes; 1: N-1; 2: N-2.
        max_relaxation = min(2, self.active_count)

        for level in range(max_relaxation + 1):
            candidates = [
                result
                for result in evaluated
                if len(result["relaxed"]) == level
            ]

            if candidates:
                self.relaxation_level = level
                self.search_details = candidates
                return [result["house"] for result in candidates]

        return []
    # ------------------------------------------------------------------
    # Mostrar preguntes sense llistes d'opcions
    # ------------------------------------------------------------------
    def print_question(self, question, possible_options=None):
        print(question["question"])

    # ------------------------------------------------------------------
    # Explicació de cada restricció relaxada
    # ------------------------------------------------------------------
    def describe_relaxation(self, house, key):
        preference = self.user_preferences[key]

        if key == "type":
            actual = {"sale": "compra", "rent": "lloguer"}.get(house.get("type"), str(house.get("type")))
            return f"Tipus: havies indicat {self.describe_preference(key, preference).lower()}, però aquest habitatge és de {actual}."
        
        if key == "income":
            income = self.normalize_number(preference)
            price = self.normalize_number(house["price"])
            limit = income * 0.35
            return f"Límit del 35%: el lloguer és de {price:g} €/mes, per sobre del límit de {limit:g} €/mes."
        
        if key in {"bedrooms", "bathrooms", "square_meters", "floor", "price"}:
            actual = self.normalize_number(house.get(key))

            if isinstance(preference, tuple):
                min_val, max_val = preference
            else:
                min_val, max_val = (None, preference) if key == "price" else (preference, None)
            
            min_val = self.normalize_number(min_val)
            max_val = self.normalize_number(max_val)
            
            reasons = []
            
            is_aprox = False
            if min_val is not None and max_val is not None:
                mid = (min_val + max_val) / 2
                if abs(min_val - mid * 0.85) < 1e-4 and abs(max_val - mid * 1.15) < 1e-4:
                    is_aprox = True
                    reasons.append(f"té {actual:g} (demanaves uns {mid:g})")
            
            if not is_aprox:
                if min_val is not None and actual < min_val:
                    reasons.append(f"té {actual:g} (demanaves mínim {min_val:g})")
                if max_val is not None and actual > max_val:
                    reasons.append(f"té {actual:g} (demanaves màxim {max_val:g})")
            
            labels = {"bedrooms": "Habitacions", "bathrooms": "Banys", "square_meters": "Superfície", "floor": "Planta", "price": "Preu"}
            return f"{labels[key]}: {' i '.join(reasons)}"
        
        labels = {"terrace": "Terrassa", "elevator": "Ascensor", "commercial_use": "Ús comercial"}
        
        if key in labels:
            actual = self.normalize_boolean(house.get(key))
            detail = "l'habitatge no disposa d'aquesta característica" if actual is False else "no consta que compleixi aquest requisit"
            return f"{labels[key]}: era un requisit obligatori, però {detail}."
        
        return self.describe_preference(key, preference)

    # ------------------------------------------------------------------
    # Presentació dels resultats
    # ------------------------------------------------------------------
    def print_suitable_houses(self, suitable_houses):
        if not suitable_houses:
            print(
                "\nNo he trobat habitatges amb les teves preferències, "
                "ni relaxant fins a dues restriccions."
            )
            return

        if self.relaxation_level == 0:
            print(
                f"\nHe trobat {len(suitable_houses)} habitatges "
                "que compleixen totes les restriccions actives."
            )
        else:
            matched = self.active_count - self.relaxation_level

            print(
                "\nNo hi ha habitatges que compleixin totes "
                "les restriccions actives."
            )
            print(
                f"Et mostro {len(suitable_houses)} alternatives "
                f"que compleixen {matched} de {self.active_count} "
                f"restriccions."
            )

            if matched == 0:
                print(
                    "Atenció: aquestes alternatives no compleixen "
                    "cap de les restriccions actives."
                )

        for detail in self.search_details:
            house = detail["house"]

            self.print_house(house)

            if detail["relaxed"]:
                print("Restriccions relaxades per a aquest habitatge:")

                for key in detail["relaxed"]:
                    print("-", self.describe_relaxation(house, key))

                print()

    # ------------------------------------------------------------------
    # Comiat
    # ------------------------------------------------------------------
    def print_goodbye(self):
        print("\nD'acord, tanquem la conversa.")
        super().print_goodbye()

    # ------------------------------------------------------------------
    # Edició individual de camps
    # ------------------------------------------------------------------
    def reask_field(self, key):
        if key == "type":
            operation = self.process_multichoice_question(
                {"question": "Vols comprar o llogar un habitatge?", "prompt": "Resposta (comprar/llogar): "},
                ["sale", "rent"]
            )
            self.user_preferences["type"] = operation
            if operation == "rent" and not self.user_preferences.get("income"):
                self.user_preferences["income"] = self.ask_income_question()
                
        elif key == "income":
            if self.user_preferences.get("type") == "rent":
                self.user_preferences["income"] = self.ask_income_question()
            else:
                print("Els ingressos només es demanen per a operacions de lloguer.")
                
        elif key == "location":
            self.print_question({"question": "En quina ciutat prefereixes viure?"})
            print("Opcions:", ", ".join(self.LOCATION_OPTIONS))
            while True:
                import re
                answer = self.read_input("Ciutat: ")
                special, value = self.handle_special_answer(answer)
                if special:
                    self.user_preferences["location"] = value
                    break
                text = self.normalize_text(answer)
                location = None
                if re.search(r"(hospitalet|hospi)", text): location = "L'Hospitalet de Llobregat"
                elif re.search(r"(esplugues|esplu)", text): location = "Esplugues de Llobregat"
                elif re.search(r"(coloma|santa coloma|santaco)", text): location = "Santa Coloma de Gramenet"
                elif re.search(r"(barcelona|bcn|barna)", text): location = "Barcelona"
                if location is not None:
                    self.user_preferences["location"] = location
                    break
                print("Escriu una de les ciutats indicades (pots posar només 'esplugues', 'bcn', etc) o «qualsevol».")
                
        elif key == "price":
            is_rent = self.user_preferences.get("type") == "rent"
            q = {
                "question": "Quin és el teu pressupost màxim mensual per al lloguer?" if is_rent else "Quin és el teu pressupost màxim per comprar l'habitatge?",
                "prompt": "Lloguer (€/mes): " if is_rent else "Pressupost de compra (€): ",
                "answer_key": "price"
            }
            self.user_preferences["price"] = self.ask_intent_question(q, default_intent="max")
            
        elif key == "bedrooms":
            self.user_preferences[key] = self.ask_intent_question({"question": "Quantes habitacions busques?", "prompt": "Habitacions: ", "answer_key": key}, "min")
        elif key == "bathrooms":
            self.user_preferences[key] = self.ask_intent_question({"question": "Quants banys necessites?", "prompt": "Banys: ", "answer_key": key}, "min")
        elif key == "square_meters":
            self.user_preferences[key] = self.ask_intent_question({"question": "Quina superfície busques en m²?", "prompt": "Superfície (m²): ", "answer_key": key}, "min")
        elif key == "floor":
            self.user_preferences[key] = self.ask_intent_question({"question": "A quina planta vols viure?", "prompt": "Planta (ex: 0 per baixa): ", "answer_key": key}, "min")
            
        elif key == "terrace":
            self.user_preferences[key] = self.ask_yes_no_any_question({"question": "És imprescindible que tingui terrassa?", "prompt": "Terrassa: "})
        elif key == "elevator":
            self.user_preferences[key] = self.ask_yes_no_any_question({"question": "És imprescindible que tingui ascensor?", "prompt": "Ascensor: "})
        elif key == "commercial_use":
            self.user_preferences[key] = self.ask_yes_no_any_question({"question": "Necessites que permeti ús comercial?", "prompt": "Ús comercial: "})


    # ------------------------------------------------------------------
    # Flux principal amb confirmació
    # ------------------------------------------------------------------
    def run(self):
        self.print_welcome()
        
        self.user_preferences = {}
        self.search_details = []
        self.relaxation_level = None
        
        # Primera vegada preguntem tot seguit
        self.ask_questions()

        while True:
            self.print_preferences_summary()

            if self.ask_confirmation("Aquestes preferències són correctes?"):
                break

            print("\nLlavors, què vols fer ara?")
            print("1. Canviar un camp concret (escriu 'habitacions', 'preu', 'ciutat', etc.)")
            print("2. Tornar a començar de zero (escriu 'tot' o 'començar')")
            print("3. Cancel·lar la cerca (escriu 'sortir')")
            
            while True:
                answer = self.read_input("\nQuè vols modificar?: ").casefold()
                
                if self.is_exit_command(answer):
                    print("\nNo executaré la cerca amb preferències no confirmades.")
                    self.print_goodbye()
                    return []
                    
                text = self.normalize_text(answer)
                
                if re.search(r"(tot|comecar|reini|zero)", text):
                    print("\nPerfecte, tornem a començar de zero.\n")
                    self.user_preferences = {}
                    self.ask_questions()
                    break # Trenca el bucle intern, torna a imprimir resum
                
                key = None
                if re.search(r"(habitacio|habitacions)", text): key = "bedrooms"
                elif re.search(r"(bany|banys)", text): key = "bathrooms"
                elif re.search(r"(preu|pressupost|lloguer|diners)", text): key = "price"
                elif re.search(r"(ciutat|ubicacio|poble)", text): key = "location"
                elif re.search(r"(superficie|metres|m2)", text): key = "square_meters"
                elif re.search(r"(planta|pis)", text): key = "floor"
                elif re.search(r"(terrassa|balco|pati)", text): key = "terrace"
                elif re.search(r"(ascensor)", text): key = "elevator"
                elif re.search(r"(comercial|local)", text): key = "commercial_use"
                elif re.search(r"(ingressos|sou)", text): key = "income"
                elif re.search(r"(tipus|comprar|llogar)", text): key = "type"
                
                if key is not None:
                    print(f"\nD'acord, anem a actualitzar aquest camp.")
                    self.reask_field(key)
                    print("\nCamp actualitzat correctament!")
                    break # Trenca el bucle intern, torna a imprimir resum
                    
                print("No he reconegut el camp. Escriu per exemple 'habitacions', 'ciutat', 'tot' per reiniciar o 'sortir'.")

        suitable_houses = self.find_suitable_houses()
        self.print_suitable_houses(suitable_houses)

        return suitable_houses

In [19]:
system = FlexibleDialogueSystem("./house_data.json")
resultats = system.run()

Benvingut/da a l'assistent de cerca d'habitatges!
Pots escriure «sortir» per acabar la conversa o «qualsevol» si no tens preferència.
Vols comprar o llogar un habitatge?
Entesos, vols comprar.
En quina ciutat prefereixes viure?
Opcions: L'Hospitalet de Llobregat, Esplugues de Llobregat, Barcelona, Santa Coloma de Gramenet
Quin és el teu pressupost màxim per comprar l'habitatge?
Quantes habitacions busques?
Quants banys necessites?
Quina superfície busques en metres quadrats?
Entesos, no fixem cap preferència per a aquest camp.
A quina planta vols viure?
Entesos, no fixem cap preferència per a aquest camp.
És imprescindible que l'habitatge tingui terrassa?
Entesos, no és un requisit: És imprescindible que l'habitatge tingui terrassa?
És imprescindible que l'edifici tingui ascensor?
Entesos, no és un requisit: És imprescindible que l'edifici tingui ascensor?
Necessites que l'habitatge permeti un ús comercial?
Entesos, no és un requisit: Necessites que l'habitatge permeti un ús comercial?